#PARTE 1
### - Extracción y Diagnóstico (Extract) 

In [0]:
"""
- De pathlib importamos Path para trabajar con rutas de carpetas y archivos

- Pandas lo usamos principalmente para generar los dataframes (tablas)

- Numpy para calculos numericos y arreglos, en mi caso lo uso más para el uso de --> np.where( condición , valor_si_cumple , valor_no_cumple )
  que es como la formula =Si( ) de excel que nos permite agregar un valor que queremos en una nueva columna del dataframe(tabla)
  si se cumple una condición y otra si es que no se cumple 
  
"""
from pathlib import Path
import pandas as pd
import numpy as np

### 1.1 - Creación "df_menus" de la carpeta restaurantes-menus.csv

In [0]:
"""
CREAR DATAFRAME DE RESTURANTES-MENUS
"""

# PASO 1: Asignamos a una variable la ruta de acceso a la carpeta de archivos
DATOS = Path("/Workspace/Users/valentinsebastian540@gmail.com/Proceso-ETL-en-Food-Delivery-con-Python/restaurant-menus.csv")
             

# PASO 2: con .glo("*") --> obtenemos las DIRECCIONES de los archivos de la carpeta DATOS, de TODOS
archivos = DATOS.glob("*")

lista_df =[]

# PASO 3: Con este sript validamos que el archivo sea tipo csv, lo convertimos en una dataframe y 
#         lo agragamos a la lista_df, donde despúes se concatenan en un solo dataframe llamado df_menus.7
#         Gracias al "ignore_index = True" vuelve a numerar las filas desde 0.
#         Sin eso, el índice repetiría números (0, 1, 2, 0, 1, 2...). 
for archivo in archivos:
    if archivo.suffix ==".csv":
        df = pd.read_csv(archivo, sep=",")
        lista_df.append(df)
    else:
        print("np es archivo tipo csv")
df_menus = pd.concat(lista_df, ignore_index = True)

In [0]:
df_menus

In [0]:
# Con info() podemos ver los detalles del dataframe que hemos creado 
# detalles como el typo de dato y cuanto de memoria usa ese dataframe
# datos que pueden ser relevantes para una optimización o alguna otra opción
df_menus.info()

In [0]:
# Con el método ".shape" podemos ver la cantidad de filas por columnas
# (filas,columnas) 
df_menus.shape

In [0]:
# Con "isnull()" nos indica de cada dato del df_ventas cual es verdadero o falso
# respondiendo la pregunta si es o no es un dato nulo --> dandonos un booleano 

# Con el uso de ".isnull().sum()" nos da la cantidad de datos nulos por tipo de columna
# util para analizar la cantidad de nulos a limpiar del df_ventas
df_menus.isnull().sum()

### 1.2 - Creación "df_restaurantes" de la carpeta restaurants.csv

In [0]:
df_restaurantes = pd.read_csv("/Workspace/Users/valentinsebastian540@gmail.com/Proceso-ETL-en-Food-Delivery-con-Python/restaurants.csv/restaurants.csv",sep=",")


In [0]:
df_restaurantes 

In [0]:
df_restaurantes.info()

In [0]:
df_restaurantes.shape

In [0]:
df_restaurantes.isnull().sum()

#PARTE 2
### - Ingeniería de Características y Limpieza (Transform)

### 2.1 - Limpieza del df_restaurantes

In [0]:
df_restaurantes

In [0]:
# Creamos una función para poder remplazar los simbolos de "$","$$", etc por categorias que nos indique
# su categoria y si no tiene este simbolo "$" ósea es null, se remplaza por "sin categoria"
def mapeo_precios(fila):
    if fila["price_range"] == "$":
        return "Económico"
    elif fila["price_range"] == "$$":
        return "Moderadamente caro"
    elif fila["price_range"] == "$$$":
        return "Caro"
    elif fila["price_range"] == "$$$$":
        return "Muy caro"
    else:
        return "sin categoria"

In [0]:
# Aqui le decimos al dataframe que en la misma columna que se selecciono, que reescriba ese valor
# con el valor que nos returna la función "mapeo_precios( )" aplicada a cada fila

df_restaurantes["price_range"] = df_restaurantes.apply(mapeo_precios, axis=1)

In [0]:
df_restaurantes

In [0]:
# Aqui dividos la columna "full_address" en 4 nuevas columnas ["street","city","zip_state","zip_code2"]
df_restaurantes[["street","city","zip_state","zip_code2"]] = df_restaurantes["full_address"].str.split(",", n=3, expand=True)

In [0]:
df_restaurantes

In [0]:
# Aqui eliminamos la columna con valores repetidos "zip_code2"
df_restaurantes=df_restaurantes.drop(columns="zip_code2")

In [0]:
# Con str.strip() quitamos los espacios en blanco de la parte de adelante y atrás del texto
df_restaurantes["city"] = df_restaurantes["city"].str.strip()
df_restaurantes["street"] = df_restaurantes["street"].str.strip()
df_restaurantes["zip_state"] = df_restaurantes["zip_state"].str.strip()

In [0]:
# Con esto nos quedamos con las filas que en "score" cumplan las condiciones de no ser nulos o ser mayor a 0
df_restaurantes=df_restaurantes[df_restaurantes["score"].notna() & (df_restaurantes["score"] != 0) ]

In [0]:
df_restaurantes["category"]=df_restaurantes["category"].str.lower()

In [0]:
df_restaurantes

### 2.2 - Limpieza del df_menus

In [0]:
df_menus


In [0]:
df_menus["price"]=df_menus["price"].apply(lambda x:float(x.replace("USD","").strip()))

In [0]:
df_menus


In [0]:
# Esto se lee asi: dejame en el df_menu las filas que sean no nulas y que sean diferentes a cero de precio
df_menus=df_menus[df_menus["price"].notna() & (df_menus["price"] !=0)]

In [0]:
df_menus

### 2.3 - Optimización y Cruce (Control de Memoria RAM): 

In [0]:
df_restaurantes

In [0]:
# Con este script nos quedamos con los restaurantes que tengan un rating mayor a 100, es como un filtro 
df_restaurantes=df_restaurantes[df_restaurantes["ratings"]>100]


In [0]:
df_restaurantes.head()

In [0]:
df_menus.head()

In [0]:
# Con esto juntamos ambos dataframes (tablas) mediante el id que tienen, el id funciona como una key de un modelo de datos
# La forma en la que se unen es mediante un inner join, de esta manera generamos la tabla con IDs similares que ambas tablas tienen
df_master=pd.merge(
            df_restaurantes,
            df_menus,
            left_on="id",
            right_on="restaurant_id",
            how="inner"
        )

In [0]:
df_restaurantes.shape

In [0]:
df_menus.shape

In [0]:
df_master.shape

In [0]:
df_master

#PARTE 3
### - Carga a Base de Datos (Load) 

In [0]:
import sqlite3
import pandas as pd


In [0]:
# crear el puente, la conexión con la base de datos entre PYTHON y la BASE DE DATOS
# Si "delivery_insights.db" no existe, se crea la base datos sola

con = sqlite3.connect("delivery_insights.db")

In [0]:
df_master.to_sql("master_food_data",con,if_exists="replace", index=False)

# "master_food_data" es el nombre del dataframe(tabla)--> "df_master" que se guarda en la base de datos "delivery_insights.db"
# con el "con" creas el puente, la conexión con la base de datos entre PYTHON y la BASE DE DATOS
# el if_exists="replace" nos dice esto: si esa tabla ya existía, borrala y creala de nuevo
# index=False --->no guarda los números de fila como columna.

#Para subir y guardar los datos a la base de datos SQL con el --> "to_sql"
#Traerlos es con el --> "read_sql"

In [0]:
query = """
    select *
    from master_food_data
    """

pd.read_sql(query,con).head()

In [0]:
df_master.info()

#PARTE 4
### - Business Intelligence: Reporte al CEO (SQL + Visualización) 

###Preguna 1  
###¿Cuáles son las 5 ciudades (ciudad) con la mayor cantidad de restaurantes consolidados?

In [0]:
"master_food_data"

query = """
select city, count(1) as cantidad_restaurantes
from master_food_data
group by city
order by count(1) DESC
limit 5
"""

pd.read_sql(query, con)

###Preguna 2
###¿Cuantos restaurantes hay por cada categoria de la columna rango de precios?

In [0]:
query = """
select price_range,
       count(1) as total_por_categoria,
       round(count(1)*100.0/sum(count(*)) over(),2) as porcentaje_por_categoria
from master_food_data
group by price_range
order by total_por_categoria desc
"""

pd.read_sql(query, con)

###Preguna 3
###Para las 3 ciudades más populares obtenidas en la pregunta 1, calcule el precio promedio de los platos en el menú. 
###¿En qué ciudad es más caro comer en promedio?

In [0]:
query = """
select city,avg(price) as precio_promedio
from master_food_data
where city in ('Milwaukee','Seattle','Lynnwood')
group by city
order by avg(price) DESC
"""
pd.read_sql(query, con)

###Preguna 4
###¿Un restaurante más caro garantiza una mejor experiencia para el usuario?
###Use score y el rango de precios(categoria) para responder

In [0]:
query ="""

SELECT name_x,
        price_range,
        CASE price_range
            when 'sin categoria' then 10
            when 'Económico' then 20
            when 'Moderadamente caro' then 30
            when 'Caro' then 40
            else 0
        END AS Puntaje_referencia_categoria,
        avg(score) as Puntaje_por_encuesta
FROM master_food_data
GROUP BY name_x, price_range
ORDER BY avg(score) DESC
"""

pd.read_sql(query, con)